# Deep learning homework: curly hair or straight hair classification
- Network with random weights, sees batches of labelled images, measures loss and calibarates its weights to reduce loss for training.
- Repeat training with data augmentation.

# 0 . Get data and verify data

In [5]:
!wget -q https://github.com/SVizor42/ML_Zoomcamp/releases/download/straight-curly-data/data.zip
!sha256sum data.zip
!unzip -q data.zip

9e53453e3017502f22860cb08558f0cdb343e102fb61feab75960616185e7d67  data.zip


In [6]:
!sha256sum data.zip #checksum is correct 

9e53453e3017502f22860cb08558f0cdb343e102fb61feab75960616185e7d67  data.zip


In [7]:
!ls data

test  train


In [8]:
!ls -R /kaggle/input/datasets/forogh/requirements-cpu-txt

/kaggle/input/datasets/forogh/requirements-cpu-txt:
requirements-cpu.txt


In [9]:
!python -m pip install -r /kaggle/input/datasets/forogh/requirements-cpu-txt/requirements-cpu.txt

Looking in indexes: https://download.pytorch.org/whl/cpu, https://pypi.org/simple
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.4/184.4 MB 118.5 MB/s  0:00:010:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 71.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 9.3 MB/s  0:00:01 eta 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 10.0 MB/s  0:00:00 eta 0:00:01
  Attempting uninstall: Pillow
    Found existing installation: pillow 13.0.0
    Uninstalling pillow-13.0.0:
      Successfully uninstalled pillow-13.0.0
  Attempting uninstall: numpy━━━━━━━━━━━━━━━━━━━ 0/4 [Pillow]
    Found existing installation: numpy 2.1.3━━━━━━━━━━━━━━━━━━━━━━ 1/4 [numpy]
    Uninstalling numpy-2.1.3:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/4 [numpy]
      Successfully uninstalled numpy-2.1.3━━━━━━━━━━━━━━━━━━━━━━━━ 1/4 [numpy]
  Attempting uninstall: torch━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/4 [numpy]
    Found existing installation: torch 2.11.0+cu130━━━━━━━━━━━

# 1. Imports and seeding

- training involves randomness, initial weights and order images are shuffled, seed ensures the randomness is repeatable


In [10]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
print(torch.__version__, np.__version__) 

2.11.0+cu130 2.1.3


In [11]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cpu")

# 2. Preprocessing, datasets and loaders
- ensures all images are the samey on size
- converts images into vector representation, RGB, h, w scaled to 0-1
- normalised (mean, std), shfits and rescales each colour channel so values are around 0, netwroks train smoothly on centered inputs,

In [12]:
from torchvision import datasets, transforms

train_transforms = transforms.Compose([
    transforms.Resize((200, 200)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

train_dataset = datasets.ImageFolder("data/train", transform=train_transforms)
print(len(train_dataset))
print(train_dataset.classes)        # expect ['curly', 'straight']
print(train_dataset.class_to_idx) 

800
['curly', 'straight']
{'curly': 0, 'straight': 1}


In [13]:
evaluation_transforms = transforms.Compose([
    transforms.Resize((200, 200)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

evaluation_dataset = datasets.ImageFolder("data/test", transform=evaluation_transforms)
print(len(evaluation_dataset), evaluation_dataset.class_to_idx)

201 {'curly': 0, 'straight': 1}


## DataLoader
- feed images to the model in batches of 20, fits in memory and updating the weights after each small batch, shuffle = True, reorders training images every epoch so model cannot learn order.
Epoch- full pass thorugh training images- 800/20= 40 weight updates per epoch

In [14]:
loader_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset,
    batch_size=20,
    shuffle=True,
    num_workers=0,
    generator=loader_generator,
)
evaluation_loader = DataLoader(
    evaluation_dataset,
    batch_size=20,
    shuffle=False,
    num_workers=0,
)


1. Conv2D- CNN, filter by 3x3 grid of weights that slides across each image, multiples by its weights by the pixels and adds them up with a number, this layer has 32 filters, learns 32 different patterns, with its own feature map.

2. ReLU- replaces every negative number with 0, keeps it non linear stacked layers would collapse into one big linear function

3. MaxPool2d- looks at each 2x2 block and keeps only the largest value, height and width- data shrinks, modle become sless sesnsitive

4. flatten- unrolls the block

5.  **`Linear(313632, 64)` + ReLU:** a **fully connected** layer. Each of the 64 outputs looks at all 313,632 inputs and combines the detected features into higher-level judgements.

6.   **`Linear(64, 1)`:** squeezes down to one number, the **logit**. Large positive means "straight", large negative means "curly", near 0 means unsure.

In [15]:
import torch.nn as nn


class HairModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=0) #convolution- a filter 3x3 grid.
        self.pool = nn.MaxPool2d(kernel_size=2) #looks at each 2x2 block and keeps its largest value
        self.hidden = nn.Linear(32 * 99 * 99, 64)
        self.output = nn.Linear(64, 1)

    def forward(self, x):
        x = self.pool(nn.functional.relu(self.conv(x)))
        x = x.flatten(start_dim=1)
        x = nn.functional.relu(self.hidden(x)) #relu function replaces every negative number with 0
        return self.output(x)

In [16]:
model = HairModel().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.002,
    momentum=0.8,
)

## ## Question 1 — loss and output contract

Which loss matches the model above?

- `nn.MSELoss()`
- `nn.BCEWithLogitsLoss()`
- `nn.CrossEntropyLoss()`
- `nn.CosineEmbeddingLoss()`

There is one intended answer: the model emits one logit and the loss applies the
stable sigmoid-plus-binary-cross-entropy operation.

answer: - `nn.BCEWithLogitsLoss()`
- model outputs one raw logit, turning into probability needs a sigmoid for two classses with binary cross entropy. BCE logit loss- does the sigmoid and cross entropy together.

## Question 2 — parameter count

What is the total number of trainable parameters? Count biases as well and use
`sum(p.numel() for p in model.parameters())`.

- `896`
- `11214912`
- `15896912`
- `20073473`


In [17]:
n_params = sum(p.numel() for p in model.parameters())
print("Q2 parameters:", n_params)      

# Optional sanity check of the shapes described above
x = torch.zeros(1, 3, 200, 200)
print(model.conv(x).shape)                        # [1, 32, 198, 198]
print(model.pool(model.conv(x)).shape) 

Q2 parameters: 20073473
torch.Size([1, 32, 198, 198])
torch.Size([1, 32, 99, 99])


## Training metrics

Train for 10 epochs with the baseline transform. For every epoch, evaluate the
held-out split with `model.eval()` and `torch.no_grad()`. Store these four
series in a JSON file named `history_baseline.json`:

```python
{
    "train_loss": [...],
    "train_accuracy": [...],
    "evaluation_loss": [...],
    "evaluation_accuracy": [...],
}
```

For accuracy, use `(torch.sigmoid(logits) >= 0.5)`. Use the number of examples,
not the number of batches, when averaging loss and accuracy.

## Question 3 — baseline accuracy

Calculate the median of `train_accuracy` in `history_baseline.json`. Submit the
value rounded to two decimal places. The grader accepts an absolute error of
`0.05`; the answer is not selected from a fixed option list.

In [18]:
def run_epoch(loader, train):
    model.train(train)
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x = x.to(device)
            y = y.float().unsqueeze(1).to(device)

            logits = model(x)
            loss = criterion(logits, y)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * x.size(0)
            preds = (torch.sigmoid(logits) >= 0.5).float()
            correct += (preds == y).sum().item()
            n += x.size(0)
    return total_loss / n, correct / n

In [20]:
def train_phase(loader, epochs=10):
    history = {
        "train_loss": [],
        "train_accuracy": [],
        "evaluation_loss": [],
        "evaluation_accuracy": [],
    }
    for epoch in range(epochs):
        tl, ta = run_epoch(loader, train=True)
        el, ea = run_epoch(evaluation_loader, train=False)
        history["train_loss"].append(tl)
        history["train_accuracy"].append(ta)
        history["evaluation_loss"].append(el)
        history["evaluation_accuracy"].append(ea)
        print(f"epoch {epoch+1:2d} | train loss {tl:.4f} acc {ta:.4f} | eval loss {el:.4f} acc {ea:.4f}")
    return history

In [23]:
import json

In [21]:
history_baseline = train_phase(train_loader, epochs=10)

epoch  1 | train loss 0.6452 acc 0.6362 | eval loss 0.6026 acc 0.6567
epoch  2 | train loss 0.5508 acc 0.7037 | eval loss 0.5892 acc 0.6617
epoch  3 | train loss 0.4986 acc 0.7525 | eval loss 0.6171 acc 0.6219
epoch  4 | train loss 0.4276 acc 0.7987 | eval loss 0.6253 acc 0.7015
epoch  5 | train loss 0.3988 acc 0.8113 | eval loss 0.6261 acc 0.6766
epoch  6 | train loss 0.3893 acc 0.8075 | eval loss 0.6106 acc 0.7114
epoch  7 | train loss 0.3686 acc 0.8275 | eval loss 0.6707 acc 0.7164
epoch  8 | train loss 0.3058 acc 0.8675 | eval loss 0.7060 acc 0.6667
epoch  9 | train loss 0.2443 acc 0.8900 | eval loss 0.7297 acc 0.7164
epoch 10 | train loss 0.1589 acc 0.9537 | eval loss 0.8826 acc 0.6617


NameError: name 'json' is not defined

In [24]:
with open("history_baseline.json", "w") as f:
    json.dump(history_baseline, f)

q3 = round(float(np.median(history_baseline["train_accuracy"])), 2)
q4 = round(float(np.std(history_baseline["train_loss"], ddof=0)), 3)
print("Q3 (median train accuracy):", q3)
print("Q4 (std of train loss):    ", q4)

Q3 (median train accuracy): 0.81
Q4 (std of train loss):     0.136


# Data augmentation
- resizing and rotating the images to transform- to get more data 

In [25]:
train_transform_augmented = transforms.Compose([
    transforms.Resize((200, 200), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.RandomRotation(
        50,
        interpolation=transforms.InterpolationMode.NEAREST,
    ),
    transforms.RandomResizedCrop(
        200,
        scale=(0.9, 1.0),
        ratio=(0.9, 1.1),
        interpolation=transforms.InterpolationMode.BILINEAR,
    ),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

train_dataset_aug = datasets.ImageFolder("data/train", transform=train_transform_augmented)
train_loader_aug = DataLoader(
    train_dataset_aug,
    batch_size=20,
    shuffle=True,
    num_workers=0,
    generator=loader_generator,   # same generator object, NOT re-seeded
)

In [26]:
history_augmented = train_phase(train_loader_aug, epochs=10)

with open("history_augmented.json", "w") as f:
    json.dump(history_augmented, f)

q5 = round(float(np.mean(history_augmented["evaluation_loss"])), 3)
q6 = round(float(np.mean(history_augmented["evaluation_accuracy"][-5:])), 2)
print("Q5 (mean eval loss):          ", q5)
print("Q6 (mean last-5 eval accuracy):", q6)

epoch  1 | train loss 0.6389 acc 0.6687 | eval loss 0.6039 acc 0.6418
epoch  2 | train loss 0.5984 acc 0.6713 | eval loss 0.5989 acc 0.6866
epoch  3 | train loss 0.6047 acc 0.6687 | eval loss 0.5708 acc 0.7164
epoch  4 | train loss 0.5590 acc 0.7075 | eval loss 0.5651 acc 0.7065
epoch  5 | train loss 0.5406 acc 0.7075 | eval loss 0.6355 acc 0.6617
epoch  6 | train loss 0.5138 acc 0.7400 | eval loss 0.5980 acc 0.6716
epoch  7 | train loss 0.5414 acc 0.7075 | eval loss 0.5896 acc 0.7114
epoch  8 | train loss 0.5220 acc 0.7350 | eval loss 0.5720 acc 0.7015
epoch  9 | train loss 0.5091 acc 0.7500 | eval loss 0.6107 acc 0.6567
epoch 10 | train loss 0.5089 acc 0.7462 | eval loss 0.5377 acc 0.7313
Q5 (mean eval loss):           0.588
Q6 (mean last-5 eval accuracy): 0.69


In [27]:
print("Q1: nn.BCEWithLogitsLoss()")
print("Q2:", n_params)
print("Q3:", q3)
print("Q4:", q4)
print("Q5:", q5)
print("Q6:", q6)

Q1: nn.BCEWithLogitsLoss()
Q2: 20073473
Q3: 0.81
Q4: 0.136
Q5: 0.588
Q6: 0.69
